# AutoSense causal feature ablation (Kaggle)

Runs the committed F0/F1/F2 causal feature definitions with fixed Random Forest/LSTM models. No validation/test data select features or settings.

In [ ]:
!pip -q uninstall -y jax jaxlib
!pip -q install tensorflow==2.18.0 keras==3.15.1 numpy==2.0.2 pandas==2.2.2 scikit-learn==1.9.0
!python -c "import sys,tensorflow as tf,sklearn,numpy,pandas; print({'python':sys.version.split()[0],'tensorflow':tf.__version__,'sklearn':sklearn.__version__,'numpy':numpy.__version__,'pandas':pandas.__version__,'devices':[str(x) for x in tf.config.list_physical_devices()]})"

In [ ]:
from pathlib import Path
import hashlib,json,shutil
INPUT=Path('/kaggle/input'); archives=list(INPUT.rglob('kaggle_feature_ablation_input_bundle.zip'))
assert len(archives)==1, f'Attach exactly one Dataset containing kaggle_feature_ablation_input_bundle.zip; found {archives}'
UNPACKED=Path('/kaggle/working/feature_ablation_input')
if UNPACKED.exists(): shutil.rmtree(UNPACKED)
shutil.unpack_archive(archives[0],UNPACKED)
candidates=[p for p in UNPACKED.rglob('feature_ablation.py') if p.parent.name=='ieee_experiments' and p.parent.parent.name=='research_v2']
assert len(candidates)==1, f'Bundle must contain exactly one feature_ablation.py; found {candidates}'
SOURCE_ROOT=candidates[0].parents[2]; WORK=Path('/kaggle/working/autosense_feature_ablation')
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(SOURCE_ROOT,WORK)
expected={'research_v2/ieee_experiments/feature_ablation.py':'4f55e57e169a03cef6069a20a436f5e3832933c7ea616835e1da3c7ed63664aa','research_v2/baseline_evaluation.py':'1981a9e35441c272b9843a7d4f2c1599f4d033040607355a6d8892cb441b4fba','research_v2/protocol.py':'d02d7e9be7883446adfe9844597c4b9c8bb02719e942bb5a69362a6aeeaa82dd','FYP ML Training and Pipeline Code/Engine Temperature/karachi_2024_vehicle_temperature_data.csv':'d6528d327f13978592bff712684343087a99942694d99f81ebda7ff28978581e','FYP ML Training and Pipeline Code/TP_Generated/tyre_pressure_2024.csv':'63270ee9d3211afc3b6c6fcca798b89c4303721caa6924c34e44d112d45688f3','FYP ML Training and Pipeline Code/Tyre Temperature/tyre_temp_2024.csv':'451301b1fef1c683e844bffdec67cc93c607836b42ce1a344219b7b080379c3a'}
actual={rel:hashlib.sha256((WORK/rel).read_bytes()).hexdigest() for rel in expected}; assert actual==expected,json.dumps({'expected':expected,'actual':actual},indent=2)
print(f'Validated {len(expected)} exact source/data hashes; writable root: {WORK}')

In [ ]:
%cd /kaggle/working/autosense_feature_ablation
!python -u research_v2/ieee_experiments/feature_ablation.py

In [ ]:
import json,platform,sys,zipfile
import pandas as pd, tensorflow as tf
OUT=WORK/'research_v2/ieee_results'; result=OUT/'feature_ablation.csv'; assert result.exists()
df=pd.read_csv(result); required={'Dataset','Dataset SHA256','Feature configuration','Model','Seed','MAE','RMSE','R2','Experiment script SHA256','Protocol SHA256','Architecture source SHA256','Model parameters','Epochs run','Best epoch','Training seconds','Unit'}
assert required <= set(df.columns), sorted(required-set(df.columns)); assert len(df)==18 and not df.duplicated(['Dataset','Feature configuration','Model','Seed']).any()
assert set(df['Dataset'])=={'engine_temperature','tyre_pressure','tyre_temperature'} and set(df['Feature configuration'])=={'F0','F1','F2'} and set(df['Model'])=={'Random Forest','LSTM'} and set(df['Seed'])=={42}
assert df['Max epochs'].eq(20).all() and df['Batch size'].eq(64).all() and df['Patience'].eq(5).all() and df['Shuffle'].eq(False).all()
assert df['Dataset SHA256'].isin(set(expected.values())).all(); assert df.loc[df['Feature configuration'].eq('F2'),'Unavailable-causal-feature policy'].str.contains('no test origins discarded').all()
manifest={'phase':'feature_ablation','input_sha256':actual,'python':sys.version,'tensorflow':tf.__version__,'platform':platform.platform(),'devices':[str(x) for x in tf.config.list_physical_devices()],'expected_rows':18,'validation':'unique Dataset/Feature configuration/Model/Seed; F0/F1/F2 fixed; script asserts split metadata and F2 keeps standard test origins'}
manifest_path=OUT/'kaggle_feature_ablation_run_manifest.json'; manifest_path.write_text(json.dumps(manifest,indent=2))
zip_path=Path('/kaggle/working/feature_ablation_results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z: z.write(result,result.name); z.write(manifest_path,manifest_path.name)
print(df.groupby(['Dataset','Feature configuration','Model'])[['MAE','RMSE','R2']].mean().round(6).to_string()); print(f'Validated 18 rows. Download: {zip_path}')